In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO
from ftfy import fix_encoding

In [ ]:
shortnametable = ""
longnametable2022 = ""
longnametable2020 = ""
longnametablealtdaten = ""
output = ""

In [ ]:
# with open(snakemake.input["shortnametable"], mode="r", encoding="iso-8859-1") as file:
#    shortnames = pd.read_excel(file)

shortnames = pd.read_excel(shortnametable)
shortnames.loc[~shortnames["Elementname BED"].isna(), "Elementname BED"] = (
    shortnames.loc[~shortnames["Elementname BED"].isna(), "Elementname BED"].apply(
        fix_encoding
    )
)


def readsoup(f):
    with open(f) as fp:
        soup = BeautifulSoup(fp)
    return soup


longnames2022 = readsoup(longnametable2022)
longnames2020 = readsoup(longnametable2020)
longnamesaltdaten = readsoup(longnametablealtdaten)

In [ ]:
# TODO 1) Fix der Excel Datei mit fillforward der NAs in der Namen Datei
shortnames = shortnames.fillna(method="ffill")

In [ ]:
shortnames.head(5)

In [ ]:
# TODO finden der Tabellen
tables = [[], [], []]
original = [longnamesaltdaten, longnames2020, longnames2022]
for i in range(len(original)):
    for tableblock in original[i].select("div.tableblock table"):
        df = pd.read_html(StringIO(str(tableblock)))[0]

        if not type(df) is pd.DataFrame:
            continue
        if "Schlüsselwort" in df:
            continue
        if "Auswahl 1" in df["Elementname"].unique():
            continue
        if "version" in df["Elementname"].unique():
            continue
        if "Fall_Nr" in df["Elementname"].unique():
            continue
        if "Beginn Auswahlelement" in df["Elementname"].unique():
            continue
        if "Medizinische_Daten" in df["Elementname"].unique():
            continue
        tables[i].append(df)

# outer join mit den kurznamen (shortname.Elementname BED - tableblock.Elementname)

describtions2019 = pd.concat(tables[0]).dropna(subset="Beschreibung")
describtions2019.insert(0, "Datensatz", "2019")
describtions2019.insert(0, "Erstes Aufkommen", "2019")

describtions2020 = pd.concat(tables[2])  # .drop_duplicates() -> keine dublikate
describtions2020.insert(0, "Datensatz", "2020")
describtions2020.insert(0, "Erstes Aufkommen", "2020")

describtions2022 = pd.concat(tables[1])  # .drop_duplicates() -> keine dublikate
describtions2022.insert(0, "Datensatz", "2022")
describtions2022.insert(0, "Erstes Aufkommen", "2022")

describtionsFull = pd.concat([describtions2019, describtions2020, describtions2022])

describtionsFull = describtionsFull.sort_values(by="Erstes Aufkommen").drop_duplicates(
    subset=describtionsFull.columns.difference(["Datensatz", "Erstes Aufkommen"]),
    keep="first",
)

merge2022 = pd.merge(
    shortnames,
    describtions2022,
    left_on="Elementname BED",
    right_on="Elementname",
    how="outer",
)
merge2020 = pd.merge(
    shortnames,
    describtions2020,
    left_on="Elementname BED",
    right_on="Elementname",
    how="outer",
)

test2022 = pd.merge(
    shortnames,
    describtions2020,
    left_on="Name der Tabelle",
    right_on="Elementname",
    how="outer",
).drop_duplicates()

# Überlappung Analyse zwischen den verschiedenen
resultCombineBeforMerge = pd.merge(
    shortnames,
    describtionsFull,
    left_on="Elementname BED",
    right_on="Elementname",
    how="outer",
).drop_duplicates()
resultCombineBeforMerge = resultCombineBeforMerge.drop_duplicates(
    subset=resultCombineBeforMerge.columns.difference(["Datensatz", "Erstes Aufkommen"])
)

### Erstes Aufkommen setzten
Hier werden alle gleichen Datensätze aus den verschiedenen Versionen gruppiert.
Es werden immer die Daten des neusten Datensatzes verwendet (siehe Spalte "Datensatz"). Jedoch wird vermerkt, wann ein Datensatz, dass erste mal erwähnt wurde (Siehe Spalte "Erstes Aufkommen").

In [ ]:
results2 = resultCombineBeforMerge.groupby("Shortnames in der BED-DB", as_index=False)


def setFirstApperance(df):
    if df.shape[0] == 1:
        return df

    fa = df.tail(1)
    fa["Erstes Aufkommen"] = df.iloc[0]["Erstes Aufkommen"]

    return fa


resultCombineBeforMerge = results2.apply(setFirstApperance).reset_index(drop=True)
df = resultCombineBeforMerge.iloc[dict(results2.indices)["EBasisBlutgrIQTIG"], :]
df

In [ ]:
resultCombineAfterMerge = pd.concat([merge2020, merge2022])

results = [
    merge2020,
    merge2022,
    resultCombineBeforMerge,
    resultCombineAfterMerge,
    test2022,
]

resultCombineAfterMerge = resultCombineAfterMerge.dropna(
    subset=["Beschreibung"]
).reset_index(drop=True)
resultCombineAfterMerge

In [ ]:
# resultCombineBeforMerge.to_excel("results/resultCombineBeforMerge.xlsx")

### Elemente denen eine Beschreibung zugewiesen werden konnte

In [ ]:
dataset = [
    "Nur 2020",
    "Nur 2022",
    "Erst 19+20+22 dann verbunden",
    "Erst verbunden dann 20+22",
    "mit Tabellenname verbunden",
]
total = []
missingInfo = []
missingShortname = []
matches = []

for index, result in enumerate(results):
    total.append(len(result))
    missingInfo.append(result["Inhalt/Form"].isna().sum())
    missingShortname.append(result["Elementname BED"].isna().sum())
    matches.append(total[index] - missingInfo[index] - missingShortname[index])

for i in range(len(total)):
    print(
        f"[{dataset[i]}] Gesamt: {total[i]} | Verbundene Einträge: {matches[i]} ({(matches[i] / total[i] * 100):.2f}%)"
    )

data = {
    "Datensatz": dataset,
    "Gesamt": total,
    "Fehlende Beschreibung": missingInfo,
    "Fehlende Kurznamen": missingShortname,
}

summary = pd.DataFrame(data)
summary

### Nicht gefundene Shortnames
Dies sind alle Elementnamen aus der Excel Datei, die keinem Beschreibungstext in den Definitionen hinterlegt hatten.
Grund dafür ist, dass es sich bei diesen Elementen um die Empfängernummern handelt und die Definition für diese werden nicht für jede Datei einzeln wiederholt.

In [ ]:
resultCombineBeforMerge.loc[resultCombineBeforMerge["Elementname"].isna(), :]

In [ ]:
resultCombineBeforMerge.loc[
    ~resultCombineBeforMerge["Elementname BED"].isna(), "Elementname BED"
].apply(fix_encoding)

In [ ]:
resultCombineBeforMerge["nanInt"] = resultCombineBeforMerge["Beschreibung"].isna()
resultCombineBeforMerge.groupby("Name der Tabelle")["nanInt"].apply(
    lambda x: x.sum() / x.size
)

### Beschreibungen anpassen
Einige Beschreibungstexte enthalten Dopplungen (z.B 'Geschlecht,Geschlecht,Geschlecht,Geschlecht') 
Diese werden hier gekürzt

In [ ]:
def formatBeschreibung(row):
    if isinstance(row, str):
        if "," in row:
            substrings = row.split(",")
            if all(s.strip() == substrings[0] for s in substrings):
                return substrings[0]
    return row

In [ ]:
resultCombineBeforMerge["Beschreibung"] = resultCombineBeforMerge["Beschreibung"].apply(
    formatBeschreibung
)

resultCombineBeforMerge

### Export

In [ ]:
resultCombineBeforMerge.to_csv(output)